# Experimento 2 · Repartir el peso del bit 5

**Pregunta:** ¿qué cambia al representar una contribución de 32 ppb mediante dos posiciones de 16 ppb?

Esta es una propuesta experimental V1 para aprender y poner a prueba la idea del asesor. Contiene una sola celda Python, independiente del ejercicio anterior. Usa el kernel **Python (lorawan11 .venv)**.

**Alcance:** enteros de 0 a 255 ppb, resolución de 1 ppb, payload de cuatro bytes y modificaciones sobre el campo del valor. El límite 255 es una decisión de este ejercicio; no es un límite físico del ozono. No sustituimos el codificador de la tesis.

Trabajamos con payloads sin cifrar. Todavía no evaluamos ataques sobre AES, datos RAMA ni detección LSTM.


## 1. Elegir los pesos

Mantenemos canal y tipo para comparar las mismas posiciones de bytes en el laboratorio. **El valor usa una semántica propia:** este payload V1 no es compatible con el decodificador actual ni con un decodificador CayenneLPP genérico. Usar el tipo 02 aquí es una convención del prototipo; una integración real requerirá identificar y acordar el formato.

| Posición del valor | Peso en la codificación actual | Peso en V1 |
|---|---:|---:|
| 0, 1, 2, 3 | 1, 2, 4, 8 | 1, 2, 4, 8 |
| 4 | 16 | 16 |
| 5 | 32 | **16** |
| 6 | 64 | 64 |
| 7 | 128 | 128 |
| 8 | 256 | **16** |
| 9–15 | Posiciones superiores; 15 es el signo | Reservadas en cero; V1 rechaza patrones que las activen |

Los bits 5 y 8 aportan juntos 32 ppb cuando ambos están encendidos. Los bits 6 y 7 siguen produciendo cambios grandes: esta versión sólo interviene el bit 5.

La tabla está definida por nosotros; no cambia la electrónica ni el peso con que un entero convencional interpreta esas posiciones. **Es el decodificador V1 quien suma los nuevos pesos.**


## 2. Representar y recuperar 160 ppb

La representación actual tiene encendidos los bits 7 y 5: `128 + 32 = 160`.

V1 conserva esos bits y enciende también el 8: `128 + 16 + 16 = 160`.

| Caso | Payload completo | Lectura correcta |
|---|---|---:|
| Actual | `01 02 00 A0` | 160 con el decodificador actual |
| V1 | `01 02 01 A0` | 160 con el decodificador V1 |
| V1 después de flip 5 | `01 02 01 80` | 144 con el decodificador V1 |

**Ojo:** el entero binario convencional `01 A0` vale 416. La medición V1 vale 160 porque aplicamos nuestros pesos. Cambiar a big-endian no introduce esta regla: sólo fija el orden de bytes.

El codificador usa una regla determinista: si el bit 5 original es 0, el par (5, 8) será (0, 0); si era 1, será (1, 1). El decodificador admite también los pares (0, 1) y (1, 0), que aportan 16. No exige que ambas posiciones coincidan.

Así, esta versión estudia el efecto de repartir pesos; **no usa el desacuerdo del par como alarma de integridad**. Los ataques sobre posiciones reservadas se rechazan por formato y no se incluyen en las 2304 pruebas de flips activos que aparecen abajo.


## 3. Ejecutar la primera comparación

La celda define el codificador y el decodificador V1. Después comprueba la recuperación exacta de los 256 valores del rango y el efecto de un flip en cada una de las nueve posiciones activas.

La comparación usa 147, 160 y 175 ppb con el umbral experimental de 155 ppb. Son ejemplos de cruce de una lectura; no representan declaraciones ni anulaciones oficiales de contingencias.

Al final genera tres muestras nuevas en `samples/` para inspeccionarlas con poke. Si encuentra una muestra que editaste, avisa y la conserva.

**Antes de ejecutar:** ¿crees que reducir el desplazamiento de 32 a 16 impedirá el cruce en los tres ejemplos?


In [ ]:
# V1 didáctica: repartir únicamente la contribución de 32 ppb.
from pathlib import Path
import sys
from IPython.display import Markdown, display

actual = Path.cwd().resolve()
RAIZ = next((p for p in (actual, *actual.parents)
             if (p / "src/encoding/cayenne.py").is_file()), None)
if RAIZ is None:
    raise RuntimeError("Abre este notebook dentro del repositorio.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.encoding.cayenne import encode_o3, decode_o3, flip_bit

# Índice de la tupla = posición física del bit, empezando en 0.
PESOS_V1 = (1, 2, 4, 8, 16, 16, 64, 128, 16)
UMBRAL_EJEMPLO = 155

def codificar_v1(ppb):
    """0..255 ppb; la contribución original de 32 ocupa los bits 5 y 8."""
    if type(ppb) is not int or not 0 <= ppb <= 255:
        raise ValueError("V1 acepta enteros de 0 a 255 ppb.")
    bit5_original = (ppb >> 5) & 1
    palabra = ppb | (bit5_original << 8)
    return bytes([1, 2]) + palabra.to_bytes(2, byteorder="big")

def decodificar_v1(payload):
    """Suma los pesos encendidos, incluso si los bits 5 y 8 difieren."""
    if len(payload) != 4 or payload[:2] != bytes([1, 2]):
        raise ValueError("Este ejemplo requiere canal 1, tipo 02 y cuatro bytes.")
    palabra = int.from_bytes(payload[2:], byteorder="big")
    if palabra >> 9:
        raise ValueError("Patrón fuera de V1: bits reservados 9..15 encendidos.")
    return sum(((palabra >> bit) & 1) * peso
               for bit, peso in enumerate(PESOS_V1))

# Comprobación exhaustiva del rango, sin datos RAMA ni entrenamiento.
for valor in range(256):
    trama = codificar_v1(valor)
    assert len(trama) == len(encode_o3(valor)) == 4
    assert decodificar_v1(trama) == valor
    for bit, peso in enumerate(PESOS_V1):
        cambio = decodificar_v1(flip_bit(trama, bit)) - valor
        assert abs(cambio) == peso
assert codificar_v1(160) == bytes.fromhex("01 02 01 A0")
assert decodificar_v1(flip_bit(codificar_v1(160), 5)) == 144
print("Verificado: 256 recuperaciones exactas y 2304 flips en bits activos.")

def cruza_umbral(original, recibido):
    return "Sí" if (original >= UMBRAL_EJEMPLO) != (recibido >= UMBRAL_EJEMPLO) else "No"

filas = [
    "| Original (ppb) | Actual: flip 5 | ¿Cruza 155? | V1: flip 5 | ¿Cruza 155? |",
    "|---:|---:|:---:|---:|:---:|",
]
for valor in (147, 160, 175):
    recibido_actual = decode_o3(flip_bit(encode_o3(valor), 5))
    recibido_v1 = decodificar_v1(flip_bit(codificar_v1(valor), 5))
    filas.append(f"| {valor} | {recibido_actual} | {cruza_umbral(valor, recibido_actual)} "
                 f"| {recibido_v1} | {cruza_umbral(valor, recibido_v1)} |")
display(Markdown("\n".join(filas)))

base_v1 = codificar_v1(160)
un_flip = flip_bit(base_v1, 5)
dos_flips = flip_bit(un_flip, 8)
for nombre, payload in [("V1 original", base_v1),
                        ("V1 flip 5", un_flip),
                        ("V1 flips 5 y 8", dos_flips)]:
    print(f"{nombre}: {payload.hex(' ').upper()} → {decodificar_v1(payload)} ppb")
print(f"Un decodificador actual leería {decode_o3(base_v1)} ppb en el original V1.")

# Muestras para GNU poke. Conserva cualquier muestra que hayas editado.
carpeta = RAIZ / "experiments/cayenne_mod/samples"
muestras = {"v1_160ppb.bin": base_v1, "v1_160ppb_flip_bit5.bin": un_flip,
            "v1_160ppb_flip_bits5_8.bin": dos_flips}
for nombre, payload in muestras.items():
    ruta = carpeta / nombre
    if ruta.exists() and ruta.read_bytes() != payload:
        raise FileExistsError(f"Muestra editada; se conserva: {ruta}")
carpeta.mkdir(parents=True, exist_ok=True)
for nombre, payload in muestras.items():
    ruta = carpeta / nombre
    if not ruta.exists():
        with ruta.open("xb") as archivo:
            archivo.write(payload)
print(f"Muestras para inspeccionar: {carpeta}")


## 4. Interpretar los resultados

| Original | Codificación actual: flip 5 | V1: flip 5 | Resultado respecto a 155 ppb |
|---:|---:|---:|---|
| 147 | 179 | 163 | Ambas fabrican un cruce hacia arriba |
| 160 | 128 | 144 | Ambas ocultan el cruce de esta lectura |
| 175 | 143 | 159 | V1 conserva la lectura por encima del umbral |

**Lo demostrado:** para el bit 5, el desplazamiento de un flip baja de 32 a 16 ppb y se conservan exactamente los valores sin ataque del rango elegido. En algunos casos se evita el cambio de decisión; en otros persiste.

**Lo que aún no sabemos:** qué sucede con la tasa global de daño en las mediciones reales, con el conjunto completo de posiciones atacables y con varios flips. El atacante conoce la distribución de pesos: no dependemos de ocultársela.

En particular, V1 añade otra posición activa de peso 16. En una representación legítima el par (5, 8) siempre es 00 o 11. Invertir ambos cambia su contribución de 0 a 32 o de 32 a 0: **en esta versión esos dos flips no se cancelan**. Para 160 ppb, invertir 5 y 8 produce 128 ppb. La cancelación de 01 ↔ 10 existe, pero esos pares no son la representación inicial que genera este codificador.

El tamaño sigue siendo de cuatro bytes. Eso verifica igualdad de longitud del payload del ejemplo; no mide tiempo de cómputo ni consumo energético.


## 5. Inspeccionar V1 con GNU poke

Desde la raíz del repositorio, después de ejecutar la celda:

```bash
poke experiments/cayenne_mod/samples/v1_160ppb.bin
```

Dentro de poke, lee la misma palabra con su interpretación binaria habitual:

```text
.set endian big
.set obase 16
dump :from 0#B :size 4#B
.set obase 2
uint<16> @ 2#B
.set obase 10
uint<16> @ 2#B
```

Debes encontrar `01 02 01 A0`, el patrón `0000000110100000` y el entero convencional **416**. Poke todavía no tiene una descripción de V1: para recuperar la medición, cuenta los bits encendidos **8, 7 y 5**, y aplica nuestros pesos: **16 + 128 + 16 = 160**.

Puedes abrir la muestra atacada en esa sesión:

```text
.file experiments/cayenne_mod/samples/v1_160ppb_flip_bit5.bin
.set obase 2
uint<16> @ 2#B
```

El patrón será `0000000110000000`: quedan encendidos 8 y 7. Con pesos V1 suma **144**, aunque como entero binario convencional vale 384.

Esta diferencia entre palabra almacenada y valor interpretado es la base de la modificación. Las operaciones de lectura usadas aquí ya se comprobaron en GNU poke 4.3 en el ejercicio anterior.


## Tus observaciones

- ¿Por qué el mismo `01 A0` puede interpretarse como 416 o como 160?
- ¿Qué regla compartida necesitan el dispositivo y el receptor?
- ¿En cuál de los tres casos V1 evitó el cruce del umbral? ¿Por qué en los otros no?
- ¿Qué obtiene el atacante si invierte los bits 5 y 8 de una medición codificada por V1?
- ¿Qué posiciones grandes siguen sin protección en este diseño?

**Mis notas:**

